# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access key properties from the metadata
metadata = dataset.metadata.to_json()
print(f"Dataset name: {metadata.get('name','')}")
print(f"Description: {metadata.get('description','')}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display summary of all record sets, fields, columns

# The Dataset object provides .record_sets() to list record sets
print("Available Record Sets:")
record_sets = list(dataset.record_sets())
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', rs_id)
    print(f"  @id: {rs_id} | Name: {rs_name}")

# Collect all fields for each record set by their @id
print("\nFields in Record Sets:")
for rs in record_sets:
    rs_id = rs['@id']
    print(f"- Record Set @id: {rs_id} - {rs.get('name',rs_id)}")
    for fld in rs.get('field', []):
        field_id = fld['@id']
        field_name = fld.get('name', field_id)
        print(f"    Field @id: {field_id} | Name: {field_name}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all record sets found in the previous cell
dataframes = dict()
record_set_ids = [rs['@id'] for rs in record_sets]

# List to show which record sets can be loaded (catch errors for those without data)
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for Record Set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
    print("-"*60)
# We'll continue with the first record set that loaded at least one record
main_record_set_id = None
for rs_id, df in dataframes.items():
    if len(df) > 0:
        main_record_set_id = rs_id
        break

if main_record_set_id is not None:
    print(f"\nProceeding with main Record Set: {main_record_set_id}")
    print("Columns available:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick a numeric field (e.g., 'coverage_percentage' or similar)
df = dataframes[main_record_set_id]
numeric_field = None
candidate_fields = [c for c in df.columns if df[c].dtype in ['float64', 'int64']]
if not candidate_fields:
    # Try to infer if there's a field that looks numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if df[c].dtype in ['float64', 'int64']:
                candidate_fields.append(c)
        except:
            continue
if candidate_fields:
    numeric_field = candidate_fields[0]
else:
    raise ValueError("No numeric field found to perform EDA.")

print(f"Using numeric field for EDA: {numeric_field}")
# Define a threshold for filtering
threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, prefer 'description' or first object dtype column
categorical_field = None
cat_candidates = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
if cat_candidates:
    categorical_field = cat_candidates[0]
if categorical_field:
    grouped_df = filtered_df.groupby(categorical_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {categorical_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we have a categorical field, plot boxplot
if categorical_field:
    plt.figure(figsize=(12,5))
    top_cats = df[categorical_field].value_counts().index[:5]  # limit to top N categories
    sns.boxplot(data=df[df[categorical_field].isin(top_cats)], x=categorical_field, y=numeric_field)
    plt.title(f"{numeric_field} by {categorical_field} (Top 5 categories)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we successfully loaded the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles from human mast cells using the `mlcroissant` library. We explored available record sets and fields by their `@id`, loaded records into DataFrames, and performed preliminary exploratory data analysis focusing on a selected numeric field. Visualizations such as histograms and boxplots provided an overview of data distributions and groupings. You can extend this notebook by analyzing additional fields, applying more advanced statistical methods, or integrating with domain-specific protein analysis tools.*